# channel-list-reverse-build — ex2: assemble decoder Sequential from channel pairs; verify spatial doubling

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `channel-list-reverse-build`. Running the final beacon cell reports progress against the `GAN: channel-list reverse build` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: channel-list reverse build` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`channel-list-reverse-build`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "channel-list-reverse-build"
DD_SUBTOPIC = "GAN: channel-list reverse build"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Assembling a decoder from channel pairs — deepening

Ex1 produced a list of `(in_c, out_c)` pairs for the decoder. Ex2 takes that list and builds an actual `nn.Sequential(ConvTranspose2d, BN, ReLU, ...)` decoder that doubles spatial resolution per block:

```python
blocks = []
for in_c, out_c in decoder_pairs:
    blocks.append(nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False))
    blocks.append(nn.BatchNorm2d(out_c))
    blocks.append(nn.ReLU(inplace=True))
decoder = nn.Sequential(*blocks)
```

**`stride=2 + kernel=4 + padding=1` doubles spatial dims.** Output size for ConvTranspose2d is `(H_in - 1) * stride - 2 * padding + kernel_size` = `(H_in - 1) * 2 - 2 + 4 = 2 * H_in`. The drill verifies this with a tiny input.

**No activation on the FINAL block.** A real DCGAN ends in Tanh — but to keep this drill focused on the assembly mechanic, every block (including the last) gets the BN + ReLU triplet. The test only checks shapes, not activation choice.

### Exercise 2 — assemble decoder Sequential from channel pairs; verify spatial doubling

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply consecutive-pair iteration over a reversed channel list to build an `nn.Sequential` decoder of ConvTranspose2d+BN+ReLU blocks whose forward doubles spatial resolution per block.
> Keywords: decoder, sequential, convtranspose, spatial-doubling, channel-pairs
> ```

**KCs targeted:** `channel-list-reverse-build`, `convtranspose-double-spatial`

Implement `ex2_build_decoder(encoder_channels)`. Given an encoder channel list (e.g. `[3, 64, 128, 256, 512]`), build the MIRROR decoder as a single `nn.Sequential`:

1. Build `decoder_channels = encoder_channels[::-1]` (slice reverse). For `[3, 64, 128, 256, 512]` -> `[512, 256, 128, 64, 3]`.
2. Build `decoder_pairs = list(zip(decoder_channels[:-1], decoder_channels[1:]))`. For the example: `[(512, 256), (256, 128), (128, 64), (64, 3)]`.
3. For each `(in_c, out_c)` in `decoder_pairs`, append THREE modules to a `blocks` list:
   - `nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False)`
   - `nn.BatchNorm2d(out_c)`
   - `nn.ReLU(inplace=True)`
4. Return `nn.Sequential(*blocks)`.

**Why `bias=False` on the ConvTranspose.** BatchNorm immediately follows and re-learns the per-channel bias as its beta. The ConvTranspose bias would be redundant + slightly wasteful.

**Use `inplace=True` on ReLU** to save memory (a DCGAN convention — saves ~10% on the activation buffer).

The test cell builds the decoder for a known encoder list and verifies (a) the layer counts are right, (b) the spatial dims exactly double per block.

In [ ]:
def ex2_build_decoder(encoder_channels: list[int]) -> nn.Module:
    """Build a Sequential decoder of ConvT+BN+ReLU blocks from reversed channels."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    # --- invariant 1: returns Sequential with the right module count ---
    encoder = [3, 64, 128, 256, 512]
    decoder = ex2_build_decoder(encoder)
    assert isinstance(decoder, nn.Sequential), f'expected Sequential, got {type(decoder).__name__}'
    # 4 decoder_pairs -> 4 blocks * 3 modules = 12 modules total
    n_modules = len(list(decoder))
    assert n_modules == 12, f'expected 12 modules (4 blocks * ConvT+BN+ReLU), got {n_modules}'

    # --- invariant 2: channel pattern of the ConvT layers is the reversed list ---
    convt_layers = [m for m in decoder if isinstance(m, nn.ConvTranspose2d)]
    assert len(convt_layers) == 4, f'expected 4 ConvTranspose2d, got {len(convt_layers)}'
    expected_pairs = [(512, 256), (256, 128), (128, 64), (64, 3)]
    for i, (in_c, out_c) in enumerate(expected_pairs):
        layer = convt_layers[i]
        assert layer.in_channels == in_c and layer.out_channels == out_c, (
            f'ConvT {i}: expected ({in_c}, {out_c}), got ({layer.in_channels}, {layer.out_channels})'
        )
        assert layer.bias is None, f'ConvT {i} must have bias=False (bias is None)'

    # --- invariant 3: spatial dim doubles per block (forward pass at H=4) ---
    x = t.zeros(1, 512, 4, 4)
    decoder.eval()
    with t.no_grad():
        y = decoder(x)
    # 4 blocks of stride-2 ConvT -> H,W go 4 -> 8 -> 16 -> 32 -> 64
    assert y.shape == (1, 3, 64, 64), f'final shape (4 doublings of 4): {y.shape}'

    # --- invariant 4: per-block doubling — probe between blocks ---
    # Extract progressive output by feeding through prefixes of the decoder.
    expected_h = [8, 16, 32, 64]
    x2 = t.zeros(1, 512, 4, 4)
    decoder.eval()
    with t.no_grad():
        cur = x2
        for block_idx in range(4):
            # 3 modules per block: ConvT, BN, ReLU
            for sub_idx in range(3):
                cur = decoder[block_idx * 3 + sub_idx](cur)
            assert cur.shape[-1] == expected_h[block_idx], (
                f'after block {block_idx}, expected H={expected_h[block_idx]}, got {cur.shape[-1]}'
            )

    # --- invariant 5: different channel lists work too ---
    small = [1, 16, 64]
    dec_small = ex2_build_decoder(small)
    # 2 pairs * 3 modules each
    assert len(list(dec_small)) == 6, f'small decoder: expected 6 modules, got {len(list(dec_small))}'
    convt_small = [m for m in dec_small if isinstance(m, nn.ConvTranspose2d)]
    assert (convt_small[0].in_channels, convt_small[0].out_channels) == (64, 16)
    assert (convt_small[1].in_channels, convt_small[1].out_channels) == (16, 1)
    # Spatial: 4 -> 8 -> 16
    x_small = t.zeros(1, 64, 4, 4)
    dec_small.eval()
    with t.no_grad():
        y_small = dec_small(x_small)
    assert y_small.shape == (1, 1, 16, 16), f'small decoder out: {y_small.shape}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_build_decoder(encoder_channels):
    import torch.nn as nn
    decoder_channels = encoder_channels[::-1]
    decoder_pairs = list(zip(decoder_channels[:-1], decoder_channels[1:]))
    blocks = []
    for in_c, out_c in decoder_pairs:
        blocks.append(nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False))
        blocks.append(nn.BatchNorm2d(out_c))
        blocks.append(nn.ReLU(inplace=True))
    return nn.Sequential(*blocks)
```

**Why stride=2, kernel=4, padding=1 specifically.** This is the DCGAN paper's recipe for exact 2x spatial expansion. The general ConvTranspose output formula is `(H_in - 1) * stride - 2 * padding + kernel_size + output_padding`. Plugging in: `(H-1) * 2 - 2 + 4 = 2 * H`. Other kernel/stride combos can also double — e.g. `kernel=2, stride=2, padding=0` — but the kernel=4 form has more receptive-field overlap, which reduces checkerboard artifacts in generated images.

**`bias=False` + BatchNorm is the universal pattern.** The BN layer's `beta` parameter subsumes whatever bias the conv would have provided. Keeping conv bias means redundant parameters and a small numerical asymmetry — measurable in benchmarks, uniformly avoided in practice.

**`inplace=True` on ReLU saves activation memory.** A 64x64 feature map at float32 is 16KB per channel — across 4 blocks of a ResNet-like net, the saved buffer can be a meaningful fraction of GPU RAM. The trade-off: in-place ops can fool autograd in unusual graph patterns, but plain ReLU is safe.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()